# Lab | LangChain Med

## Objectives

- continue on with lesson 2' example, use different datasets to test what we did in class. Some datasets are suggested in the notebook but feel free to scout other datasets on HuggingFace or Kaggle.
- Find another model on Hugging Face and compare it.
- Modify the prompt to fit your selected dataset.

In [ ]:
import numpy as np
import pandas as pd

## Load the Dataset
As you can see the notebook is ready to work with three different Datasets. Just uncomment the lines of the Dataset you want to use.

I selected Datasets with News. Two of them have just a brief decription of the news, but the other contains the full text.

As we are working in a free and limited space, I limited the number of news to use with the variable MAX_NEWS. Feel free to pull more if you have memory available.

The name of the field containing the text of the new is stored in the variable *DOCUMENT* and the metadata in *TOPIC*

In [ ]:
# news = pd.read_csv('/kaggle/input/topic-labeled-news-dataset/labelled_newscatcher_dataset.csv', sep=';')
# MAX_NEWS = 1000
# DOCUMENT="title"
# TOPIC="topic"

#news = pd.read_csv('/kaggle/input/bbc-news/bbc_news.csv')
#MAX_NEWS = 1000
#DOCUMENT="description"
#TOPIC="title"


news = pd.read_csv('datasets/articles.csv')
MAX_NEWS = 100
DOCUMENT="Article Body"
TOPIC="Article Header"

ChromaDB requires that the data has a unique identifier. We can make it with this statement, which will create a new column called **Id**.


In [ ]:
news["id"] = news.index
news.head()

,Unnamed: 0,Published Date,Author,Source,Article Header,Sub_Headings,Article Body,Url,id
0,0,"July 7, 2023",Adam Zewe,MIT News Office,Learning the language of molecules to predict ...,This AI system only needs a small amount of da...,['Discovering new materials and drugs typicall...,https://news.mit.edu/2023/learning-language-mo...,0
1,1,"July 6, 2023",Alex Ouyang,Abdul Latif Jameel Clinic for Machine Learning...,MIT scientists build a system that can generat...,"BioAutoMATED, an open-source, automated machin...",['Is it possible to build machine-learning mod...,https://news.mit.edu/2023/bioautomated-open-so...,1
2,2,"June 30, 2023",Jennifer Michalowski,McGovern Institute for Brain Research,"When computer vision works more like a brain, ...",Training artificial neural networks with data ...,"['From cameras to self-driving cars, many of t...",https://news.mit.edu/2023/when-computer-vision...,2
3,3,"June 30, 2023",Mary Beth Gallagher,School of Engineering,Educating national security leaders on artific...,"Experts from MIT’s School of Engineering, Schw...",['Understanding artificial intelligence and ho...,https://news.mit.edu/2023/educating-national-s...,3
4,4,"June 30, 2023",Adam Zewe,MIT News Office,Researchers teach an AI to write better chart ...,A new dataset can help scientists develop auto...,['Chart captions that explain complex trends a...,https://news.mit.edu/2023/researchers-chart-ca...,4


In [ ]:
#Because it is just a course we select a small portion of News.
subset_news = news.head(MAX_NEWS)

## Import and configure the Vector Database
I'm going to use ChromaDB, the most popular OpenSource embedding Database.

First we need to import ChromaDB, and after that import the **Settings** class from **chromadb.config** module. This class allows us to change the setting for the ChromaDB system, and customize its behavior.

In [ ]:
!pip install chromadb

In [ ]:
import chromadb
from chromadb.config import Settings

Now we need to create the seetings object calling the **Settings** function imported previously. We store the object in the variable **settings_chroma**.

Is necessary to inform two parameters
* chroma_db_impl. Here we specify the database implementation and the format how store the data. I choose ***duckdb***, because his high-performace. It operate primarly in memory. And is fully compatible with SQL. The store format ***parquet*** is good for tabular data. With good compression rates and performance.

* persist_directory: It just contains the directory where the data will be stored. Is possible work without a directory and the data will be stored in memory without persistece, but Kaggle dosn't support that.

In [ ]:
chroma_client = chromadb.PersistentClient(path="/path/to/persist/directory")

## Filling and Querying the ChromaDB Database
The Data in ChromaDB is stored in collections. If the collection exist we need to delete it.

In the next lines, we are creating the collection by calling the ***create_collection*** function in the ***chroma_client*** created above.

In [ ]:
collection_name = "news_collection"
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
        chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)


It's time to add the data to the collection. Using the function ***add*** we need to inform, at least ***documents***, ***metadatas*** and ***ids***.
* In the **document** we store the big text, it's a different column in each Dataset.
* In **metadatas**, we can informa a list of topics.
* In **id** we need to inform an unique identificator for each row. It MUST be unique! I'm creating the ID using the range of MAX_NEWS.


In [ ]:
import pandas as pd

# Example data
MAX_NEWS = 5
TOPIC = "topic"
DOCUMENT = "content"

# Suppose this is your news dataset
news_df = pd.DataFrame({
    DOCUMENT: [
        "NASA launches a new satellite.",
        "SpaceX prepares for next launch.",
        "NASA's budget increases for 2025.",
        "Mars rover sends new photos.",
        "NASA partners with private companies."
    ],
    TOPIC: [
        "NASA", "SpaceX", "NASA", "NASA", "NASA"
    ]
})

# Subset the news (e.g., take first MAX_NEWS rows)
subset_news = news_df.head(MAX_NEWS)


In [ ]:
import pandas as pd

# Example data
MAX_NEWS = 5
TOPIC = "topic"
DOCUMENT = "content"

# Suppose this is your news dataset
news_df = pd.DataFrame({
    DOCUMENT: [
        "NASA launches a new satellite.",
        "SpaceX prepares for next launch.",
        "NASA's budget increases for 2025.",
        "Mars rover sends new photos.",
        "NASA partners with private companies."
    ],
    TOPIC: [
        "NASA", "SpaceX", "NASA", "NASA", "NASA"
    ]
})

# Subset the news (e.g., take first MAX_NEWS rows)
subset_news = news_df.head(MAX_NEWS)


In [ ]:

collection.add(
    documents=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)]
)

C:\Users\Lovely\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:15<00:00, 5.28MiB/s]


In [ ]:
results = collection.query(query_texts=["laptop"], n_results=10 )

print(results)

{'ids': [['id0', 'id4', 'id3', 'id2', 'id1']], 'embeddings': None, 'documents': [['NASA launches a new satellite.', 'NASA partners with private companies.', 'Mars rover sends new photos.', "NASA's budget increases for 2025.", 'SpaceX prepares for next launch.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'topic': 'NASA'}, {'topic': 'NASA'}, {'topic': 'NASA'}, {'topic': 'NASA'}, {'topic': 'SpaceX'}]], 'distances': [[1.6307284832000732, 1.7505826950073242, 1.803128957748413, 1.8264570236206055, 1.892703890800476]]}


## Vector MAP

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:

getado = collection.get(ids="id141",
                       include=["documents", "embeddings"])


In [ ]:
word_vectors = getado["embeddings"]
word_list = getado["documents"]
word_vectors

array([], dtype=float64)

Once we have our information inside the Database we can query It, and ask for data that matches our needs. The search is done inside the content of the document, and it dosn't look for the exact word, or phrase. The results will be based on the similarity between the search terms and the content of documents.

The metadata is not used in the search, but they can be utilized for filtering or refining the results after the initial search.


## Loading the model and creating the prompt
TRANSFORMERS!!
Time to use the library **transformers**, the most famous library from [hugging face](https://huggingface.co/) for working with language models.

We are importing:
* **Autotokenizer**: It is a utility class for tokenizing text inputs that are compatible with various pre-trained language models.
* **AutoModelForCasualLLM**: it provides an interface to pre-trained language models specifically designed for language generation tasks using causal language modeling (e.g., GPT models), or the model used in this notebook ***databricks/dolly-v2-3b***.
* **pipeline**: provides a simple interface for performing various natural language processing (NLP) tasks, such as text generation (our case) or text classification.

The model selected is [dolly-v2-3b](https://huggingface.co/databricks/dolly-v2-3b), the smallest Dolly model. It have 3billion paramaters, more than enough for our sample, and works much better than GPT2.

Please, feel free to test [different Models](https://huggingface.co/models?pipeline_tag=text-generation&sort=trending), you need to search for NLP models trained for text-generation. My recomendation is choose "small" models, or we will run out of memory in kaggle.  


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "databricks/dolly-v2-3b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)



tokenizer_config.json:   0%|          | 0.00/450 [00:00<?, ?B/s]

c:\Users\Lovely\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lovely\.cache\huggingface\hub\models--databricks--dolly-v2-3b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/819 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/5.68G [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/5.68G [00:00<?, ?B/s]

The next step is to initialize the pipeline using the objects created above.

The model's response is limited to 256 tokens, for this project I'm not interested in a longer response, but it can easily be extended to whatever length you want.

Setting ***device_map*** to ***auto*** we are instructing the model to automaticaly select the most appropiate device: CPU or GPU for processing the text generation.  

In [ ]:
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    device_map="auto",
)

Device set to use cpu


## Creating the extended prompt
To create the prompt we use the result from query the Vector Database  and the sentence introduced by the user.

The prompt have two parts, the **relevant context** that is the information recovered from the database and the **user's question**.

We only need to join the two parts together to create the prompt that we are going to send to the model.

You can limit the lenght of the context passed to the model, because we can get some Memory problems with one of the datasets that contains a realy large text in the document part.

In [ ]:
question = "Can I buy a Toshiba laptop?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
#context = context[0:5120]
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
prompt_template

"Relevant context: #NASA launches a new satellite. #NASA partners with private companies. #Mars rover sends new photos. #NASA's budget increases for 2025. #SpaceX prepares for next launch.\n\n The user's question: Can I buy a Toshiba laptop?"

Now all that remains is to send the prompt to the model and wait for its response!


In [2]:
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

NameError: name 'pipe' is not defined

# Task
Explain the error in the selected code, fix it by using a smaller model, and demonstrate the fix by running the code.

## Explain the cause of the error

### Subtask:
Explain that the `NameError` is due to the failure of the preceding cells to execute, likely caused by memory limitations when loading a large model.


**Reasoning**:
Explain the root cause of the NameError and the likely reason for the preceding cell's failure.



In [2]:
# The NameError occurred because the `pipe` object was not defined.
# Looking at the preceding code cells, the cell attempting to create the `pipe` object (cell_id: 7660416b)
# failed to execute. This failure is likely due to memory limitations when trying to load the
# "databricks/dolly-v2-3b" model, which is quite large (3 billion parameters), exceeding the available memory in the environment.
# As a result, the `pipe` object was never successfully created, leading to the NameError when it was called in the subsequent cell.

## Suggest a smaller model

### Subtask:
Suggest a smaller model from Hugging Face that is suitable for text generation and can fit within the available memory.


**Reasoning**:
Search Hugging Face for smaller text generation models that are suitable for the task and can fit within the available memory.



In [3]:
# Searching Hugging Face for smaller text generation models.
# Models with fewer parameters than 3 billion are preferred.
# Some potential smaller models for text generation include:
# - google/gemma-2b: A 2 billion parameter model from Google.
# - microsoft/phi-2: A 2.7 billion parameter model from Microsoft.
# - EleutherAI/gpt-neo-125m: A 125 million parameter model.
# - facebook/opt-125m: A 125 million parameter model.
# - Salesforce/xgen-tokenizer: A 160 million parameter model.

# Considering the available memory and the need for text generation,
# 'google/gemma-2b' or 'microsoft/phi-2' could be good options,
# but they might still be too large depending on the exact environment.
# 'EleutherAI/gpt-neo-125m', 'facebook/opt-125m', or 'Salesforce/xgen-tokenizer'
# are significantly smaller and more likely to fit within limited memory.

# Let's select 'EleutherAI/gpt-neo-125m' as a suitable smaller model for demonstration purposes.
smaller_model_id = "EleutherAI/gpt-neo-125m"
print(f"Selected smaller model: {smaller_model_id}")

Selected smaller model: EleutherAI/gpt-neo-125m


## Modify cells to use the smaller model

### Subtask:
Update the code to load the smaller model and its tokenizer.


**Reasoning**:
Update the code cell to use the smaller model ID and load the corresponding tokenizer and model.



In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "EleutherAI/gpt-neo-125m"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

## Re-run the pipeline cell

### Subtask:
Execute the cell that creates the `pipe` object using the newly loaded smaller model and tokenizer.


**Reasoning**:
The previous steps have successfully loaded the smaller model and its tokenizer. Now, execute the cell that initializes the text generation pipeline using these updated objects to create the `pipe` object.



In [5]:
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    device_map="auto",
)

Device set to use cpu


## Re-run the text generation cell

### Subtask:
Execute the cell that uses the `pipe` object to generate text using the updated pipeline with the smaller model.


**Reasoning**:
The pipeline object is now successfully created with the smaller model. The next step is to execute the cell that uses this pipeline to generate text and observe the output.



In [6]:
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

NameError: name 'prompt_template' is not defined

**Reasoning**:
The error indicates that `prompt_template` is not defined. Looking at the notebook, the cell defining `prompt_template` (cell_id: 574efc79) was executed previously. This suggests a state issue in the notebook where the variable is no longer in memory. To fix this, I need to re-execute the cell that defines `prompt_template` before attempting to use it.



In [7]:
question = "Can I buy a Toshiba laptop?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
#context = context[0:5120]
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
print(prompt_template)

NameError: name 'results' is not defined

**Reasoning**:
The error indicates that `results` is not defined. Looking at the notebook, the cell defining `results` (cell_id: ff6a4fcf) was executed previously. Similar to the `prompt_template` issue, this suggests a state issue in the notebook. To fix this, I need to re-execute the cell that defines `results` before attempting to use it to create `prompt_template`.



In [8]:
results = collection.query(query_texts=["laptop"], n_results=10 )

print(results)

NameError: name 'collection' is not defined

**Reasoning**:
The error indicates that `collection` is not defined. Looking at the notebook, the cell defining `collection` (cell_id: a0cc5748) was executed previously. This is another state issue where the variable is no longer in memory. To fix this, I need to re-execute the cells that create the `chroma_client` and define the `collection` before attempting to query it.



In [9]:
chroma_client = chromadb.PersistentClient(path="/path/to/persist/directory")

collection_name = "news_collection"
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
        chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)

# Re-adding data to the collection since it was deleted and recreated
collection.add(
    documents=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)]
)

results = collection.query(query_texts=["laptop"], n_results=10 )

print(results)

NameError: name 'chromadb' is not defined

**Reasoning**:
The error indicates that `chromadb` is not defined. This means the import statement for `chromadb` was not executed or the kernel state was reset. I need to re-import `chromadb` and `Settings` before attempting to create the client and collection again. After importing, I will recreate the client and collection, re-add the data, query the collection, redefine `results`, redefine `prompt_template`, and finally execute the cell to generate text using the pipeline. This sequence addresses all the `NameError`s encountered so far and brings the notebook back to a state where the text generation can proceed.



In [10]:
import chromadb
from chromadb.config import Settings

chroma_client = chromadb.PersistentClient(path="/path/to/persist/directory")

collection_name = "news_collection"
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
        chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)

# Re-adding data to the collection since it was deleted and recreated
collection.add(
    documents=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)]
)

results = collection.query(query_texts=["laptop"], n_results=10 )

print(results)

question = "Can I buy a Toshiba laptop?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
#context = context[0:5120]
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
print(prompt_template)

lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

ModuleNotFoundError: No module named 'chromadb'

**Reasoning**:
The error indicates that the `chromadb` module is not found, even though I attempted to re-import it. Looking back at the notebook, there was a cell (cell_id: 9dc0a6f5) that used `!pip install chromadb`. This suggests the package might not be installed or the kernel environment was reset in a way that removed installed packages. I need to re-run the pip install command before attempting to import and use `chromadb` again. After the installation, I will proceed with the rest of the steps to recreate the client and collection, re-add data, query, define the prompt, and generate text.



In [11]:
!pip install chromadb
import chromadb
from chromadb.config import Settings

chroma_client = chromadb.PersistentClient(path="/path/to/persist/directory")

collection_name = "news_collection"
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
        chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)

# Re-adding data to the collection since it was deleted and recreated
collection.add(
    documents=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)]
)

results = collection.query(query_texts=["laptop"], n_results=10 )

print(results)

question = "Can I buy a Toshiba laptop?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
#context = context[0:5120]
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
print(prompt_template)

lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.8/510.8 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 90.5 MB/s eta 0:0

NameError: name 'subset_news' is not defined

**Reasoning**:
The error indicates that `subset_news` is not defined. Looking at the notebook, the cells defining `subset_news` and the related variables `MAX_NEWS`, `DOCUMENT`, and `TOPIC` were executed previously (cell_id: 1849922b and cell_id: a5492e3f). This is another state issue where these variables are no longer in memory. To fix this, I need to re-execute the cells that define these variables and `subset_news` before attempting to add data to the collection. After defining these variables, I will proceed with the rest of the steps to recreate the client and collection, re-add data, query, define the prompt, and generate text.



In [12]:
# Re-defining variables and subset_news
MAX_NEWS = 100
DOCUMENT="Article Body"
TOPIC="Article Header"
subset_news = news.head(MAX_NEWS)

# Re-create chroma_client and collection, add data, query, define prompt, and generate text
import chromadb
from chromadb.config import Settings

chroma_client = chromadb.PersistentClient(path="/path/to/persist/directory")

collection_name = "news_collection"
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
        chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)

# Re-adding data to the collection since it was deleted and recreated
collection.add(
    documents=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)]
)

results = collection.query(query_texts=["laptop"], n_results=10 )

print(results)

question = "Can I buy a Toshiba laptop?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
#context = context[0:5120]
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
print(prompt_template)

lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

NameError: name 'news' is not defined

**Reasoning**:
The error indicates that `news` is not defined. Looking at the notebook, the cell that loads the `news` dataframe (cell_id: e70a47e0) was executed previously. This is another state issue. To fix this, I need to re-execute the cell that loads the `news` dataframe and the cell that adds the 'id' column (cell_id: 61c151df) before attempting to create `subset_news`. After loading and preparing the `news` dataframe, I will proceed with the rest of the steps to define variables, create the ChromaDB client and collection, re-add data, query, define the prompt, and generate text using the pipeline.



In [1]:
# Re-load the news dataframe and add the id column
news = pd.read_csv('datasets/articles.csv')
news["id"] = news.index

# Re-defining variables and subset_news
MAX_NEWS = 100
DOCUMENT="Article Body"
TOPIC="Article Header"
subset_news = news.head(MAX_NEWS)

# Re-create chroma_client and collection, add data, query, define prompt, and generate text
import chromadb
from chromadb.config import Settings

chroma_client = chromadb.PersistentClient(path="/path/to/persist/directory")

collection_name = "news_collection"
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
        chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)

# Re-adding data to the collection since it was deleted and recreated
collection.add(
    documents=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)]
)

results = collection.query(query_texts=["laptop"], n_results=10 )

print(results)

question = "Can I buy a Toshiba laptop?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
#context = context[0:5120]
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
print(prompt_template)

lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

NameError: name 'pd' is not defined

## Summary:

### Data Analysis Key Findings

*   The initial `NameError` was caused by the `pipe` object not being defined, which in turn was due to the preceding cell failing to execute. This failure was attributed to memory limitations when attempting to load the large "databricks/dolly-v2-3b" model.
*   A smaller model, "EleutherAI/gpt-neo-125m" (125 million parameters), was selected as a suitable alternative to the large "databricks/dolly-v2-3b" model (3 billion parameters) to mitigate memory issues.
*   The code was successfully modified to load the selected smaller model and its tokenizer.
*   The text generation pipeline (`pipe`) was successfully initialized using the smaller model and tokenizer.
*   Subsequent attempts to re-run the text generation cell failed due to a series of `NameError` and `ModuleNotFoundError` issues, indicating that necessary variables, dataframes, and libraries were not defined or imported in the execution environment.

### Insights or Next Steps

*   Ensure all necessary libraries (e.g., pandas, chromadb) are imported at the beginning of the script or notebook to avoid `ModuleNotFoundError`.
*   Verify that all required variables and dataframes (`news`, `subset_news`, `collection`, `results`, `prompt_template`) are defined and available in the current scope before attempting to use them to prevent `NameError`.
